In [1]:
# Mount Google Drive and Set Paths
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

Mounted at /content/drive


In [4]:
# Setup and load

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

base_dir = Path("/content/drive/MyDrive/gdsc-drug-response")
data_path = base_dir / "processed" / "cisplatin_rnaseq_modelready_clean.csv"
df = pd.read_csv(data_path)

df.shape
df.head()

,model_id,drug_name,ln_ic50,A1BG,A1BG-AS1,A1CF,A2M,A2M-AS1,A2ML1,A2ML1-AS1,...,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11AP1,ZYG11B,ZYX,ZZEF1,ZZZ3
0,SIDM01132,Cisplatin,3.168266,5.18545,2.9833,0.0000,0.52325,0.0000,0.0213,0.0,...,6.17065,0.31285,1.60475,3.10825,0.6801,0.0,2.58955,5.95435,3.47415,4.0301
1,SIDM00848,Cisplatin,-0.007561,4.86890,3.4476,0.0000,1.99640,0.1243,3.1890,0.0,...,7.09150,1.68260,2.33060,5.18630,0.0841,0.0,3.09930,5.52420,3.60640,4.5211
2,SIDM00269,Cisplatin,0.045420,4.54970,3.8001,0.0841,3.21570,0.0566,0.5753,0.0,...,4.53920,2.82980,3.41280,6.78270,2.3190,0.0,4.21410,9.01050,4.93400,5.4561
3,SIDM00203,Cisplatin,0.923136,4.22500,3.2095,0.0144,0.50590,0.0976,2.8156,0.0,...,6.45100,2.49570,3.47640,5.30710,3.0321,0.0,4.47310,6.95290,4.35900,5.3366
4,SIDM01111,Cisplatin,-0.134330,4.49250,2.8115,0.0144,0.47510,0.0286,0.5558,0.0,...,7.37920,1.28690,2.26000,4.94810,2.7634,0.0,3.20950,3.99550,4.02240,4.9635


In [5]:
# Define columns:

model_col    = "model_id"
drug_col     = "drug_name"
response_col = "ln_ic50"

non_feature_cols = [model_col, drug_col, response_col]
feature_cols = [c for c in df.columns if c not in non_feature_cols]

# Fill NaNs in features with the column mean
df_imputed = df.copy()
df_imputed[feature_cols] = df_imputed[feature_cols].fillna(df_imputed[feature_cols].mean())

# Optionally, confirm no NaNs remain
df_imputed[feature_cols].isna().any().sum()  # should be 0

X = df_imputed[feature_cols].values
y = df_imputed[response_col].values

X.shape, y.shape

((743, 36438), (743,))

In [6]:
#  Train/test split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

X_train.shape, X_test.shape

((594, 36438), (149, 36438))

In [13]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

ridge_a100 = Ridge(alpha=100, random_state=42)
ridge_a100.fit(X_train, y_train)

ridge_a10000 = Ridge(alpha=10000, random_state=42)
ridge_a10000.fit(X_train, y_train)

rf3 = RandomForestRegressor(
    n_estimators=300,
    max_depth=7,
    max_features=200,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42
)
rf3.fit(X_train, y_train)

RandomForestRegressor(max_depth=7, max_features=200, min_samples_leaf=5,
                      n_estimators=300, n_jobs=-1, random_state=42)

In [10]:
def regression_metrics(y_true, y_pred):
    r2   = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    return r2, rmse, mae

In [11]:
# Build the metrics table

def get_metrics(name, model):
    y_train_pred = model.predict(X_train)
    y_test_pred  = model.predict(X_test)
    r2_train, rmse_train, mae_train = regression_metrics(y_train, y_train_pred)
    r2_test,  rmse_test,  mae_test  = regression_metrics(y_test,  y_test_pred)
    return {
        "model": name,
        "r2_train": r2_train,
        "r2_test": r2_test,
        "rmse_train": rmse_train,
        "rmse_test": rmse_test,
        "mae_train": mae_train,
        "mae_test": mae_test
    }

rows = [
    get_metrics("Ridge_a100",   ridge_a100),
    get_metrics("Ridge_a10000", ridge_a10000),
    get_metrics("RF3",          rf3),
]

metrics_df = pd.DataFrame(rows)
metrics_df

,model,r2_train,r2_test,rmse_train,rmse_test,mae_train,mae_test
0,Ridge_a100,0.999871,0.304829,0.020759,1.569340,0.013543,1.230876
1,Ridge_a10000,0.870578,0.351877,0.658441,1.515305,0.511565,1.199445
2,RF3,0.763314,0.278504,0.890428,1.598778,0.684118,1.267228
